# TB-Trust — 03: Uncertainty methods and deferral (Phase 3)

Trains the evidential and deep-ensemble variants so all the uncertainty signals are comparable on the same held-out clinic, then runs the two robustness extensions: worst-case degradation search and sequential/CUSUM deferral.

`tbtrust-eval` already compares `confidence` / `mc_dropout` / `head` head to head on identical calibrated probabilities — this notebook adds the variants that need their own training runs.

In [ ]:
# --- configuration ---------------------------------------------------------
# Defaults are the Kaggle paths. Every path is read from the environment first,
# so the same notebook runs unmodified on Kaggle, locally, or in CI -- which is
# also what lets these notebooks be executed as a test rather than only read.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(WORK, exist_ok=True)
print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running a notebook is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()
print("tbtrust ready from", REPO)

In [ ]:
import subprocess
import sys


def run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    print((r.stdout or "")[-2500:])
    if r.returncode != 0:
        print((r.stderr or "")[-3000:])
        raise RuntimeError(f"command failed: {' '.join(str(c) for c in cmd)}")


TRAIN = [sys.executable, "-m", "tbtrust.train.loop"]
EVAL = [sys.executable, "-m", "tbtrust.eval.run"]

## 1. Evidential deep learning

One forward pass, no sampling: uncertainty is the Dirichlet vacuity. This is the featured calibration-focused head for the low-compute setting.

In [ ]:
run([*TRAIN, "--config", "configs/evidential_montgomery.yaml",
             f"data.manifest={MANIFEST}", f"train.output_dir={OUT}/evidential"])
run([*EVAL, "--config", "configs/evidential_montgomery.yaml",
            "--checkpoint", f"{OUT}/evidential/montgomery/best.ckpt",
            f"data.manifest={MANIFEST}"])

## 2. Deep ensemble

The calibration upper bound, not the deployed model: N models means N× storage and N× inference. Its job is to show how much the cheap methods give up.

In [ ]:
run([sys.executable, "scripts/train_ensemble.py",
     "--config", "configs/loco_montgomery.yaml", "--n-members", "3",
     f"data.manifest={MANIFEST}", f"train.output_dir={OUT}"])

In [ ]:
from pathlib import Path

from torch.utils.data import DataLoader

from tbtrust.config import load_experiment
from tbtrust.data import manifest as M
from tbtrust.data.dataset import TBDataset
from tbtrust.data.splits import split_from_config
from tbtrust.models.ensemble import DeepEnsemble, evaluate_ensemble

cfg = load_experiment("configs/loco_montgomery.yaml", overrides=[f"data.manifest={MANIFEST}"])

members = sorted(str(p) for p in Path(OUT, "ensemble", "montgomery").glob("member_*/montgomery/best.ckpt"))
assert members, "no ensemble members found -- did the previous cell run?"
ensemble = DeepEnsemble.load(cfg, members)

# split_from_config (not a bare leave_one_clinic_out) so this split is identical
# to the one the members were trained under -- otherwise "test" here could
# contain images they were trained on. Seed the dataset so the degradation is
# reproducible rather than re-randomised on every fetch.
df = split_from_config(M.load(MANIFEST), cfg)
test_ds = TBDataset(df, split="test", image_size=cfg["data"]["image_size"],
                    degradation_severity=cfg["eval"]["primary_severity"], seed=cfg["seed"])
print(f"{len(members)}-member ensemble on {len(test_ds)} held-out images:")
print(evaluate_ensemble(ensemble, DataLoader(test_ds, batch_size=cfg["train"]["batch_size"])))

Feed those member checkpoints to `tbtrust-eval --ensemble-checkpoints` to get the ensemble's AURC alongside the other signals, all on the same calibrated probabilities.

In [ ]:
run([*EVAL, "--config", "configs/loco_montgomery.yaml",
            "--checkpoint", f"{OUT}/baseline/montgomery/best.ckpt",
            "--ensemble-checkpoints", ",".join(members),
            f"data.manifest={MANIFEST}"])

## 3. Worst-case (adversarial) degradation search

The degradation pipeline is non-differentiable, so this is an honest worst-of-N-query black-box search, not a gradient attack. The number that matters: does predicted uncertainty rise specifically on the images the search makes harder, rather than uniformly?

In [ ]:
run([sys.executable, "scripts/evaluate_adversarial_robustness.py",
     "--config", "configs/loco_montgomery.yaml",
     "--checkpoint", f"{OUT}/baseline/montgomery/best.ckpt",
     "--severity", "0.7", "--n-trials", "8", "--sample-n", "20",
     f"data.manifest={MANIFEST}"])

## 4. Sequential and CUSUM deferral

Two surveillance-literature extensions on top of the per-image threshold: adaptive-stopping MC-dropout (spend fewer passes on easy cases) and a CUSUM chart that detects sustained drift in a clinic's capture quality — something per-image deferral cannot see, because it only ever looks at one image.

In [ ]:
from torch.utils.data import DataLoader

from tbtrust.config import load_experiment
from tbtrust.data import manifest as M
from tbtrust.data.dataset import TBDataset
from tbtrust.data.splits import split_from_config
from tbtrust.eval.sequential_deferral import sequential_mc_dropout_decide
from tbtrust.models.baseline import build_model
from tbtrust.utils.io import load_checkpoint

# Rebuilt from scratch so this cell does not depend on earlier cells' variables.
# Note the *baseline* checkpoint: build_model(cfg) only matches baseline-shaped weights.
cfg = load_experiment("configs/loco_montgomery.yaml", overrides=[f"data.manifest={MANIFEST}"])
model = build_model(cfg)
load_checkpoint(model, f"{OUT}/baseline/montgomery/best.ckpt")

df = split_from_config(M.load(MANIFEST), cfg)
test_ds = TBDataset(df, split="test", image_size=cfg["data"]["image_size"],
                    degradation_severity=cfg["eval"]["primary_severity"], seed=cfg["seed"])
batch = next(iter(DataLoader(test_ds, batch_size=8)))   # {'image','label','uncertainty_target','severity'}

result = sequential_mc_dropout_decide(model, batch["image"][:1], passes_min=3, passes_max=20)
print("sequential decision:", result)

In [ ]:
import numpy as np

from tbtrust.eval.sequential_deferral import CUSUMMonitor

# A clinic whose capture quality degrades partway through the day: uncertainty
# drifts up and stays up. CUSUM should flag the sustained shift, not the noise.
rng = np.random.default_rng(0)
stream = np.concatenate([rng.normal(0.20, 0.02, 20), rng.normal(0.34, 0.02, 20)])

monitor = CUSUMMonitor(target=0.20, slack=0.05, threshold=0.5)
first_alarm = None
for i, score in enumerate(stream):
    if monitor.update(float(score))["alarm_high"] and first_alarm is None:
        first_alarm = i
print("drift injected at reading 20; CUSUM first alarmed at reading", first_alarm)

Next: **04_full_evaluation_and_results.ipynb**.